In [3]:
import requests
from pathlib import Path
url = (    
"https://www.immd.gov.hk/opendata/eng/"    
"transport/immigration_clearance/"    
"statistics_on_daily_passenger_traffic.csv"
)
response = requests.get(url, timeout=60)
response.raise_for_status()
folder = Path("data/raw")
folder.mkdir(parents=True, exist_ok=True)
file_path = folder / "immd_daily.csv"
file_path.write_bytes(response.content)
print("Download completed!")
print("Saved to:", file_path)

Download completed!
Saved to: data\raw\immd_daily.csv


In [4]:
import pandas as pd
df = pd.read_csv("data/raw/immd_daily.csv")
print(df.head())
print(df.columns.tolist())
print(df.shape)

         Date                   Control Point Arrival / Departure  \
0  01-01-2021                         Airport             Arrival   
1  01-01-2021                         Airport           Departure   
2  01-01-2021  Express Rail Link West Kowloon             Arrival   
3  01-01-2021  Express Rail Link West Kowloon           Departure   
4  01-01-2021                        Hung Hom             Arrival   

   Hong Kong Residents  Mainland Visitors  Other Visitors  Total  Unnamed: 7  
0                  341                  0               9    350         NaN  
1                  803                 17              28    848         NaN  
2                    0                  0               0      0         NaN  
3                    0                  0               0      0         NaN  
4                    0                  0               0      0         NaN  
['Date', 'Control Point', 'Arrival / Departure', 'Hong Kong Residents', 'Mainland Visitors', 'Other Visitors', 

In [5]:
print("All control points:")
print(df["Control Point"].unique())
print("\nArrival and departure directions:")
print(df["Arrival / Departure"].unique())
print("\nEarliest date:")
print(df["Date"].iloc[0])
print("\nLatest date:")
print(df["Date"].iloc[-1])

All control points:
<StringArray>
[                       'Airport', 'Express Rail Link West Kowloon',
                       'Hung Hom',                          'Lo Wu',
          'Lok Ma Chau Spur Line',                 'Heung Yuen Wai',
  'Hong Kong-Zhuhai-Macao Bridge',                    'Lok Ma Chau',
                     'Man Kam To',                    'Sha Tau Kok',
                   'Shenzhen Bay',           'China Ferry Terminal',
                'Harbour Control',        'Kai Tak Cruise Terminal',
           'Macau Ferry Terminal',        'Tuen Mun Ferry Terminal',
           'Macao Ferry Terminal',                      'Huanggang']
Length: 18, dtype: str

Arrival and departure directions:
<StringArray>
['Arrival', 'Departure']
Length: 2, dtype: str

Earliest date:
01-01-2021

Latest date:
31-10-2026


In [ ]:
df["Date"] = pd.to_datetime(    
df["Date"],    
format="%d-%m-%Y"
)
start_date = "2023-02-06"
end_date = "2026-10-31"
northbound_ports = [
    "Lo Wu",
    "Lok Ma Chau Spur Line",
    "Lok Ma Chau",
    "Huanggang",
    "Shenzhen Bay",
    "Heung Yuen Wai",
    "Hong Kong-Zhuhai-Macao Bridge",
    "Express Rail Link West Kowloon"
]
northbound = df[
    (df["Date"] >= start_date) &
    (df["Date"] <= end_date) &
    (df["Control Point"].isin(northbound_ports)) &
    (df["Arrival / Departure"] == "Departure")
].copy()
northbound["Control Point"] = northbound["Control Point"].replace({
    "Lok Ma Chau": "Lok Ma Chau / Huanggang",
    "Huanggang": "Lok Ma Chau / Huanggang"
})
print("First five rows after filtering:")
print(northbound.head())
print("\nFiltered data shape:")
print(northbound.shape)
print("\nIncluded control points:")
print(northbound["Control Point"].unique())

First five rows after filtering:
            Date                   Control Point Arrival / Departure  \
23789 2023-02-06  Express Rail Link West Kowloon           Departure   
23793 2023-02-06                           Lo Wu           Departure   
23795 2023-02-06           Lok Ma Chau Spur Line           Departure   
23797 2023-02-06                  Heung Yuen Wai           Departure   
23799 2023-02-06   Hong Kong-Zhuhai-Macao Bridge           Departure   

       Hong Kong Residents  Mainland Visitors  Other Visitors  Total  \
23789                 7532               3837             378  11747   
23793                29686               3828             142  33656   
23795                23312               4419             155  27886   
23797                 2813                363               6   3182   
23799                12584                901            2310  15795   

       Unnamed: 7  
23789         NaN  
23793         NaN  
23795         NaN  
23797         NaN  
2

In [7]:
print("Number of rows after filtering:")
print(len(northbound))
print("\nControl points actually included:")
print(northbound["Control Point"].unique())
print("\nNumber of rows by control point:")
print(northbound["Control Point"].value_counts())
print("\nNumber of days covered by the study period:")
print(northbound["Date"].nunique())

Number of rows after filtering:
9529

Control points actually included:
<StringArray>
['Express Rail Link West Kowloon',                          'Lo Wu',
          'Lok Ma Chau Spur Line',                 'Heung Yuen Wai',
  'Hong Kong-Zhuhai-Macao Bridge',                    'Lok Ma Chau',
                   'Shenzhen Bay']
Length: 7, dtype: str

Number of rows by control point:
Control Point
Express Rail Link West Kowloon    1364
Lo Wu                             1364
Lok Ma Chau Spur Line             1364
Heung Yuen Wai                    1364
Hong Kong-Zhuhai-Macao Bridge     1364
Shenzhen Bay                      1364
Lok Ma Chau                       1345
Name: count, dtype: int64

Number of days covered by the study period:
1364


In [8]:
daily_northbound = (    
northbound.groupby("Date")["Hong Kong Residents"]    
.sum()    
.reset_index()
)
daily_northbound.columns = [    
"Date",    
"Northbound_Passengers"
]
print("Daily northbound trips by Hong Kong residents:")
print(daily_northbound.head(10))
print("\nTotal number of days:")
print(len(daily_northbound))
print("\nAverage daily northbound passenger count:")
print(daily_northbound["Northbound_Passengers"].mean())

Daily northbound trips by Hong Kong residents:
        Date  Northbound_Passengers
0 2023-02-06                  91029
1 2023-02-07                  84887
2 2023-02-08                  88115
3 2023-02-09                  93738
4 2023-02-10                 133457
5 2023-02-11                 168102
6 2023-02-12                 111750
7 2023-02-13                 106066
8 2023-02-14                 101605
9 2023-02-15                 102414

Total number of days:
1364

Average daily northbound passenger count:
251783.53225806452


In [9]:
from pathlib import Path
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)
daily_northbound.to_csv(    
output_folder / "daily_northbound.csv",    
index=False,    
encoding="utf-8-sig"
)
print("Y1 data saved successfully!")

Y1 data saved successfully!


In [10]:
port_daily = (    
northbound.groupby(["Date", "Control Point"])[        
"Hong Kong Residents"
    ]    
.sum()    
.reset_index()
)
port_daily = port_daily.merge(    
daily_northbound,    
on="Date",    
how="left"
)
port_daily["Port_Share"] = (    
port_daily["Hong Kong Residents"] /    
port_daily["Northbound_Passengers"]
)
print("Y2: Daily passenger share by control point")
print(port_daily.head(15))
print("\nTotal number of rows:")
print(len(port_daily))

Y2: Daily passenger share by control point
         Date                   Control Point  Hong Kong Residents  \
0  2023-02-06  Express Rail Link West Kowloon                 7532   
1  2023-02-06                  Heung Yuen Wai                 2813   
2  2023-02-06   Hong Kong-Zhuhai-Macao Bridge                12584   
3  2023-02-06                           Lo Wu                29686   
4  2023-02-06                     Lok Ma Chau                 3899   
5  2023-02-06           Lok Ma Chau Spur Line                23312   
6  2023-02-06                    Shenzhen Bay                11203   
7  2023-02-07  Express Rail Link West Kowloon                 6111   
8  2023-02-07                  Heung Yuen Wai                 3316   
9  2023-02-07   Hong Kong-Zhuhai-Macao Bridge                11821   
10 2023-02-07                           Lo Wu                28787   
11 2023-02-07                     Lok Ma Chau                 3404   
12 2023-02-07           Lok Ma Chau Spur Line  

In [13]:
share_check = (    
port_daily.groupby("Date")["Port_Share"]    
.sum()
)
print("Total port share for the first ten days:")
print(share_check.head(10))
import numpy as np
print("\nDo the port shares sum to 100% for every date?:")
print(np.allclose(share_check, 1))

Total port share for the first ten days:
Date
2023-02-06    1.0
2023-02-07    1.0
2023-02-08    1.0
2023-02-09    1.0
2023-02-10    1.0
2023-02-11    1.0
2023-02-12    1.0
2023-02-13    1.0
2023-02-14    1.0
2023-02-15    1.0
Name: Port_Share, dtype: float64

Do the port shares sum to 100% for every date?:
True


In [14]:
port_daily.to_csv(    
output_folder / "port_daily.csv",    
index=False,    
encoding="utf-8-sig"
)
print("Y2 data saved successfully!")

Y2 data saved successfully!


In [15]:
mainland_arrival = df[
    (df["Date"] >= "2023-02-06") &
    (df["Date"] <= "2026-10-31") &
    (df["Control Point"].isin(northbound_ports)) &
    (df["Arrival / Departure"] == "Arrival")
].copy()
daily_mainland = (    
mainland_arrival.groupby("Date")["Mainland Visitors"]    
.sum()    
.reset_index()
)
daily_mainland.columns = [    
"Date",    
"Mainland_Arrivals"
]
print("Y3: Daily southbound arrivals of Mainland visitors")
print(daily_mainland.head(10))
print("\nTotal number of days:")
print(len(daily_mainland))
print("\nAverage daily southbound arrivals:")
print(daily_mainland["Mainland_Arrivals"].mean())

Y3: Daily southbound arrivals of Mainland visitors
        Date  Mainland_Arrivals
0 2023-02-06              17754
1 2023-02-07              18914
2 2023-02-08              21853
3 2023-02-09              23258
4 2023-02-10              32859
5 2023-02-11              43245
6 2023-02-12              36188
7 2023-02-13              31435
8 2023-02-14              29351
9 2023-02-15              33659

Total number of days:
1364

Average daily southbound arrivals:
85269.57184750734


In [16]:
hk_all = df[
    (df["Date"] >= "2023-02-06") &
    (df["Date"] <= "2026-10-31") &
    (df["Control Point"].isin(northbound_ports))
].copy()
hk_direction = (    
hk_all.groupby(
        ["Date", "Arrival / Departure"]
    )["Hong Kong Residents"]    
.sum()    
.unstack(fill_value=0)
)
hk_direction["Net_Departure"] = (    
hk_direction["Departure"] -    
hk_direction["Arrival"]
)
daily_net = (    
hk_direction[["Net_Departure"]]    
.reset_index()
)
print("Y4: Daily net departures of Hong Kong residents")
print(daily_net.head(10))
print("\nTotal number of days:")
print(len(daily_net))

Y4: Daily net departures of Hong Kong residents
Arrival / Departure       Date  Net_Departure
0                   2023-02-06           5617
1                   2023-02-07           -891
2                   2023-02-08            -61
3                   2023-02-09           5081
4                   2023-02-10          36305
5                   2023-02-11          58906
6                   2023-02-12         -85752
7                   2023-02-13          -9092
8                   2023-02-14            678
9                   2023-02-15          -4439

Total number of days:
1364


In [17]:
master_passenger = (    
daily_northbound    
.merge(daily_mainland, on="Date", how="left")    
.merge(daily_net, on="Date", how="left")
)
print(master_passenger.head(10))
print("\nTotal rows and columns:")
print(master_passenger.shape)
master_passenger.to_csv(    
output_folder / "passenger_daily.csv",    
index=False,    
encoding="utf-8-sig"
)
print("\nDaily passenger summary saved successfully!")

        Date  Northbound_Passengers  Mainland_Arrivals  Net_Departure
0 2023-02-06                  91029              17754           5617
1 2023-02-07                  84887              18914           -891
2 2023-02-08                  88115              21853            -61
3 2023-02-09                  93738              23258           5081
4 2023-02-10                 133457              32859          36305
5 2023-02-11                 168102              43245          58906
6 2023-02-12                 111750              36188         -85752
7 2023-02-13                 106066              31435          -9092
8 2023-02-14                 101605              29351            678
9 2023-02-15                 102414              33659          -4439

Total rows and columns:
(1364, 4)

Daily passenger summary saved successfully!


In [18]:
print("1. Missing values:")
print(master_passenger.isna().sum())
print("\n2. Duplicate dates:")
print(master_passenger["Date"].duplicated().sum())
print("\n3. Descriptive statistics for each variable:")
print(master_passenger.describe())
print("\n4. Negative passenger counts:")
print(
    (master_passenger[
        ["Northbound_Passengers", "Mainland_Arrivals"]
    ] < 0).sum()
)

1. Missing values:
Date                     0
Northbound_Passengers    0
Mainland_Arrivals        0
Net_Departure            0
dtype: int64

2. Duplicate dates:
0

3. Descriptive statistics for each variable:
                      Date  Northbound_Passengers  Mainland_Arrivals  \
count                 1364            1364.000000        1364.000000   
mean   2024-12-18 12:00:00          251783.532258       85269.571848   
min    2023-02-06 00:00:00            6649.000000        1744.000000   
25%    2024-01-12 18:00:00          198713.750000       63157.500000   
50%    2024-12-18 12:00:00          243066.000000       76949.500000   
75%    2025-11-24 06:00:00          290525.500000       99585.000000   
max    2026-10-31 00:00:00          636449.000000      256340.000000   
std                    NaN           82814.541895       32707.724327   

       Net_Departure  
count    1364.000000  
mean      254.585044  
min   -457436.000000  
25%    -24444.250000  
50%       956.000000  
75% 

In [19]:
lowest_net = master_passenger.nsmallest(
    10, "Net_Departure"
)

print("Ten days with the lowest net departures:")
print(lowest_net.to_string(index=False))

highest_northbound = master_passenger.nlargest(
    10, "Northbound_Passengers"
)

print("\nTen days with the highest northbound passenger counts:")
print(highest_northbound.to_string(index=False))

print("\nMissing-value counts for all columns:")
print(master_passenger.isna().sum().to_string())


Ten days with the lowest net departures:
      Date  Northbound_Passengers  Mainland_Arrivals  Net_Departure
2026-04-07                 201740              56068        -457436
2026-05-25                 281368              71748        -387236
2025-04-21                 193487              62774        -379210
2026-10-19                 215335              71917        -374113
2024-07-01                 193888              66161        -368277
2023-10-23                 196675              58537        -317799
2026-09-27                 226890              95514        -310864
2024-06-10                 197038              66831        -301615
2024-10-13                 191030              83588        -301546
2026-06-21                 233734              84217        -292902

Ten days with the highest northbound passenger counts:
      Date  Northbound_Passengers  Mainland_Arrivals  Net_Departure
2025-04-18                 636449              61375         384003
2026-04-03         

In [11]:

check = df[
    (df["Date"] >= "2026-10-10") &
    (df["Date"] <= "2026-10-15") &
    (df["Control Point"].isin([
        "Lok Ma Chau",
        "Huanggang"
    ]))
].copy()

print(
    check[
        [
            "Date",
            "Control Point",
            "Arrival / Departure",
            "Hong Kong Residents"
        ]
    ].to_string(index=False)
)


      Date Control Point Arrival / Departure  Hong Kong Residents
2026-10-10   Lok Ma Chau             Arrival                17844
2026-10-10   Lok Ma Chau           Departure                19625
2026-10-11   Lok Ma Chau             Arrival                25547
2026-10-11   Lok Ma Chau           Departure                14836
2026-10-12     Huanggang             Arrival                27597
2026-10-12     Huanggang           Departure                30354
2026-10-12   Lok Ma Chau             Arrival                 5726
2026-10-12   Lok Ma Chau           Departure                 2438
2026-10-13     Huanggang             Arrival                33374
2026-10-13     Huanggang           Departure                35451
2026-10-14     Huanggang             Arrival                32480
2026-10-14     Huanggang           Departure                34683
2026-10-15     Huanggang             Arrival                33652
2026-10-15     Huanggang           Departure                36150
